In [ ]:
#1. 문제정의
환자의 당뇨병 여부를 예측하시오.
예측할 컬럼: Outcome(0: 정상, 1: 당뇨병)
학습용 데이터(train)을 이용해 환자의 당뇨병을 예측하는 모델을 만든 후 이를 평가용  데이터(test)에 적용해 얻은 예측 값을 다음과 같은 형식의 csv 파일로 생성하시오.
제출 파일은 다음 1개의 컬럼을 포함해야 한다.
     1) pred: 예측 값(당뇨병일 확률)
     2) 제출 파일명: ‘result.csv’

# 2. 라이브러리 및 데이터 불러오기
import pandas as pd
# train = pd.read_csv("diabetes_train.csv")
# test = pd.read_csv("diabetes_test.csv")
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch6/diabetes_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch6/diabetes_test.csv")

# 3. 탐색적 데이터 분석(EDA)
# (1)데이터 크기 확인
print("===== 데이터 크기 =====")
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)
print("\n") # 줄 바꿈

# (2)컬럼별 자료형 확인
print("===== 데이터 정보(자료형) =====")
print(train.info())
print("\n")

 (3)결측치 확인
print("===== train 결측치 수 =====")
print(train.isnull().sum().sum())
# 데이터프레임 전체의 결측치(NaN/Null) 개수를 하나의 숫자로 얻어내기 위함
print("\n")

print("===== test 결측치 수 =====")
print(test.isnull().sum().sum())
print("\n")

# (4)target 확인 -> 분류 group 개수 확인
print("===== target 빈도 =====")
print(train[‘Outcome'].value_counts())
# Series 객체에 적용되며, 해당 Series 내부에 있는 고유한 값(Unique Values)들의 빈도수를 계산

# 4. 데이터 전처리
# 원핫인코딩 (target컬럼이 범주형일 때, 필요)
# 결측치 없으므로 pass
target = train.pop(‘Outcome’)



-분류 문제에서 원-핫 인코딩(One-Hot Encoding)을 사용하는 이유는 머신러닝 모델이 범주형 데이터를 효과적으로 이해하고 처리할 수 있도록 만들기 위함.
-원-핫 인코딩을 사용하는 이유:
머신러닝 모델, 특히 수학적 계산을 기반으로 하는 모델(예: 선형 회귀, 신경망)은 숫자 형태로 된 입력 데이터를 기대.
-순서나 크기의 오해를 없애기 위해 원-핫 인코딩은 이 문제를 해결합니다. 각 범주를 독립된 특성(Feature)으로 만들고, 해당 범주에 속하면 1, 속하지 않으면 0을 부여.
-pd.get_dummies() 함수는 데이터프레임(DataFrame)에서 범주형(Categorical) 특성을 자동으로 찾아서 숫자형 특성(Dummy/Indicator Variables)으로 변환해 주는 역할


# 5. 검증 데이터 분할
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

print("\n ===== 분할된 데이터 크기 =====")
print(X_tr.shape, X_val.shape, y_tr.shape, y_val.shape)

# 6. 머신러닝 학습 및 평가
# 분류를 위한 지도학습에서 데이터를 학습시키고 예측하는 가장 기본적인 Scikit-learn 표준 파이프라인으로 랜덤 포레스트 분류기(RandomForestClassifier) 모델을 사용함.
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=0)
rf.fit(X_tr, y_tr)
pred = rf.predict_proba(X_val)

# 7. 예측 및 결과 파일 생성
pred = rf.predict_proba(test)
submit = pd.DataFrame({'pred:’pred[:,1]})
submit.to_csv("result.csv", index=False)

# 제출파일 확인
print("\n ===== 제출파일 (샘플 5개) =====")
print(pd.read_csv("result.csv").head())

In [ ]:
1. 결측치 (Missing Values) 점검
코드: df.isna().sum().sort_values(ascending=False)

역할: 데이터프레임의 각 변수(열)마다 비어있는 데이터(NaN)가 몇 개인지 세어보고, 결측치가 많은 순서대로 내림차순 정렬하여 보여줍니다.

용도 (왜 쓰는가?): * 데이터 품질 평가: 구멍 난 데이터가 어디에 얼마나 있는지 파악합니다.

전처리 의사결정: 머신러닝 모델은 빈칸을 이해하지 못합니다. 결측치가 너무 많은 열은 아예 삭제할지, 아니면 평균값이나 중앙값 등으로 채워 넣을지(대체법) 결정하는 근거가 됩니다.


2. 자료형 (Data Types) 파악 및 분리
코드: df.dtypes / select_dtypes(include/exclude="number")

역할: 변수들이 숫자(int, float)인지 문자(object)인지 확인하고, 숫자형 변수들과 비숫자형(범주형) 변수들의 이름만 따로 묶어서 추출합니다.

용도 (왜 쓰는가?): * 맞춤형 데이터 가공: 변수의 성격에 따라 전처리 방식이 완전히 달라지기 때문에 이를 분리하기 위해 씁니다.

예를 들어, 숫자형 데이터는 값의 크기를 맞추는 스케일링(정규화/표준화)을 해야 하고, 문자형(범주형) 데이터는 컴퓨터가 계산할 수 있도록 0과 1의 숫자로 바꿔주는 작업(원핫 인코딩 등)을 거쳐야 합니다.


3. 기초통계 (Descriptive Statistics) 요약
코드: df.describe(include="all")

역할: 데이터의 전반적인 요약 정보를 한눈에 보여주는 '성적표' 같은 기능입니다. (include="all"을 쓰면 숫자뿐만 아니라 문자 데이터까지 전부 요약해 줍니다.)

숫자형: 개수, 평균, 표준편차, 최솟값, 최댓값, 사분위수(25%, 50%, 75%)

문자형: 고윳값(종류) 개수, 가장 자주 나온 값(최빈값), 최빈값의 빈도수

용도 (왜 쓰는가?):

데이터 분포 및 이상치 파악: 데이터가 평균을 중심으로 잘 모여 있는지, 아니면 비정상적으로 크거나 작은 '이상치(Outlier)'가 섞여 있는지 빠르게 눈치챌 수 있습니다.

특징 추출: 범주형 데이터의 종류가 몇 개나 되는지 파악하여, 분석에 유의미한 변수인지 직관적으로 판단할 수 있게 돕습니다.


4. 타깃 분포 (Target Distribution) 확인
코드: df[TARGET].value_counts(...) / normalize=True

역할: 우리가 최종적으로 예측해야 하는 정답지(Target)의 데이터가 어떻게 구성되어 있는지 개수와 비율(%)을 확인합니다. dropna=False를 써서 정답이 비어있는(결측치) 데이터가 있는지도 함께 체크합니다.

용도 (왜 쓰는가?): * 데이터 불균형(Imbalance) 파악: 예를 들어 암 환자를 예측하는 과제인데, 정상인 데이터가 99%이고 암 환자 데이터가 1%라면 모델은 무조건 "정상"이라고만 찍어도 99점(정확도)을 받게 됩니다.

모델링 전략 수정: 이렇게 비율이 크게 차이 난다면 개수가 적은 데이터를 부풀리거나(오버샘플링), 평가지표를 단순 '정확도(Accuracy)'가 아닌 다른 지표(F1-score 등)로 바꿔야 한다는 중요한 의사결정을 내릴 수 있습니다.

정답 결측치 처리: 타깃(정답) 자체가 누락된 데이터는 학습에 방해만 되므로 보통 아예 삭제해 버립니다.


5. 변수 간 상관관계 (Correlation) 분석
코드: corr() / sns.heatmap(...)

역할: 숫자형 변수들이 서로 얼마나 밀접하게 변하는지 **상관계수(-1부터 1 사이의 값)**로 계산하고, 이를 파란색과 빨간색 같은 색상 온도 차이(히트맵)로 한눈에 보여줍니다.

용도 (왜 쓰는가?):

핵심 변수(Feature) 찾기: 타깃(Target)과 상관계수가 높은 변수를 찾습니다. 정답과 관련성이 높은 데이터일수록 모델이 예측을 잘하게 만드는 '알짜배기 힌트'가 됩니다.

다중공선성(Multicollinearity) 방지: 타깃이 아닌 일반 변수들끼리 상관관계가 너무 높은 경우(예: '태어난 연도'와 '나이')를 찾아냅니다. 둘은 사실상 똑같은 의미를 담고 있으므로 모델에 혼란을 주거나 계산 시간만 낭비하게 됩니다. 따라서 둘 중 하나는 과감히 제거(Drop)하는 전처리 기준이 됩니다.